In [35]:
from huggingface_hub import hf_hub_download
import importlib.util

# Descarga el loader unificado (está en cualquier repo)
loader_path = hf_hub_download("Reynier/dga-cnn", "dga_loader.py")
spec = importlib.util.spec_from_file_location("dga_loader", loader_path)
mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)

name_model="bilbo"
# Carga cualquier modelo
model, m = mod.load_dga_model(name_model)
results = mod.predict_domains(m, model, ["google.com", "xkr3f9mq.ru"])
print(results)

bilbo_best.pth:   0%|          | 0.00/1.58M [00:00<?, ?B/s]

model.py: 0.00B [00:00, ?B/s]

  bilbo ready.
[{'domain': 'google.com', 'label': 'legit', 'score': 0.0543}, {'domain': 'xkr3f9mq.ru', 'label': 'dga', 'score': 0.9637}]


In [36]:
# Para un solo dominio, pasamos el dominio en una lista
results = mod.predict_domains(m, model, ["google.com"])

# Como devuelve una lista, accedemos al primer elemento [0]
prediccion = results[0]

print(f"Dominio: {prediccion['domain']}")
print(f"Etiqueta: {prediccion['label']}")
print(f"Puntaje: {prediccion['score']}")

Dominio: google.com
Etiqueta: legit
Puntaje: 0.0543


In [37]:
import pandas as pd
import time

families = ['bazarbackdoor.gz',
            'bazarbackdoor_v2.gz',
            'bazarbackdoor_v3.gz',
            'bigviktor.gz',
            'bumblebee.gz',
            'ccleaner.gz',
            'dmsniff.gz',
            'enviserv.gz',
            'fosniw.gz',
            'goz.gz',
            'gozi_rfc4343.gz',
            'infy.gz',
            'monerodownloader.gz',
            'newgoz.gz',
            'ngioweb.gz',
            'pandabanker.gz',
            'pizd.gz',
            'reconyc.gz',
            'sharkbot.gz',
            'szribi.gz',
            'torpig.gz',
            'tufik.gz',
            'verblecon.gz',
            'wd.gz',
            'xshellghost.gz',
           ]

runs = 30
chunk_size = 50
offset_legit = 1500

if model is None:
    print("❌ No se pudo cargar el modelo. Deteniendo evaluación.")
else:
    print("🚀 Iniciando evaluación del modelo...")

    for family in families:
        print(f"📂 Evaluando familia: {family}")

        dga = pd.read_csv(
            f'/content/drive/My Drive/New_Families/{family}',
            chunksize=chunk_size
        )

        legit = pd.read_csv(
            '/content/drive/My Drive/Familias_Test/legit.gz',
            skiprows=range(1, offset_legit + 1),
            chunksize=chunk_size
        )

        for run in range(runs):
            print(f'  🔄 {run+1:02}/{runs}', end='\r')

            try:
                dfw = pd.concat([dga.get_chunk(), legit.get_chunk()], ignore_index=True)
            except StopIteration:
                print(f"\n⚠️ No hay suficientes datos para completar {family}")
                break

            pred = []
            prob = []
            query_time = []

            for domain_to_check in dfw.domain.values:
                st = time.time()

                try:
                    results = mod.predict_domains(m, model, [domain_to_check])
                    result = results[0]

                    label_value = 1 if result['label'] == 'dga' else 0
                    probability = result['score']

                    pred.append(label_value)
                    prob.append(probability)

                except Exception:
                    pred.append(0)
                    prob.append(0.5)

                query_time.append(time.time() - st)

            dfw['pred'] = pred
            dfw['prob'] = prob
            dfw['query_time'] = query_time

            dfw.to_csv(
                f'/content/drive/My Drive/results/results_{name_model}_{family}_{run}.csv.gz',
                index=False
            )

    print("\n✅ Evaluación completada!")

🚀 Iniciando evaluación del modelo...
📂 Evaluando familia: bazarbackdoor.gz
📂 Evaluando familia: bazarbackdoor_v2.gz
📂 Evaluando familia: bazarbackdoor_v3.gz
📂 Evaluando familia: bigviktor.gz
📂 Evaluando familia: bumblebee.gz
📂 Evaluando familia: ccleaner.gz
📂 Evaluando familia: dmsniff.gz
📂 Evaluando familia: enviserv.gz
📂 Evaluando familia: fosniw.gz
📂 Evaluando familia: goz.gz
📂 Evaluando familia: gozi_rfc4343.gz
📂 Evaluando familia: infy.gz
📂 Evaluando familia: monerodownloader.gz
📂 Evaluando familia: newgoz.gz
📂 Evaluando familia: ngioweb.gz
📂 Evaluando familia: pandabanker.gz
📂 Evaluando familia: pizd.gz
📂 Evaluando familia: reconyc.gz
📂 Evaluando familia: sharkbot.gz
📂 Evaluando familia: szribi.gz
📂 Evaluando familia: torpig.gz
📂 Evaluando familia: tufik.gz
📂 Evaluando familia: verblecon.gz
📂 Evaluando familia: wd.gz
📂 Evaluando familia: xshellghost.gz

✅ Evaluación completada!


In [38]:
import os
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix
)
import matplotlib.pyplot as plt
import seaborn as sns

# Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Función para calcular FPR y TPR
def fpr_tpr(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    fpr = fp / (fp + tn + 1e-10)
    tpr = tp / (tp + fn + 1e-10)
    return fpr, tpr

families = ['bazarbackdoor.gz', 'bazarbackdoor_v2.gz', 'bazarbackdoor_v3.gz', 'bigviktor.gz', 'bumblebee.gz', 'ccleaner.gz', 'dmsniff.gz', 'enviserv.gz', 'fosniw.gz', 'goz.gz', 'gozi_rfc4343.gz', 'infy.gz', 'monerodownloader.gz', 'newgoz.gz', 'ngioweb.gz', 'pandabanker.gz', 'pizd.gz', 'reconyc.gz', 'sharkbot.gz', 'szribi.gz', 'torpig.gz', 'tufik.gz', 'verblecon.gz', 'wd.gz', 'xshellghost.gz']
runs = 30

# Listas para métricas globales
all_acc, all_acc_std = [], []
all_pre, all_pre_std = [], []
all_rec, all_rec_std = [], []
all_f1, all_f1_std = [], []
all_fpr, all_fpr_std = [], []
all_tpr, all_tpr_std = [], []
all_qt, all_qts = [], []
total_unknowns_global = 0

for family in families:
    acc, pre, rec, f1, fpr, tpr, qt = [], [], [], [], [], [], []
    total_unknowns = 0

    for run in range(runs):
        path = f'/content/drive/My Drive/results/results_{name_model}_{family}_{run}.csv.gz'
        if not os.path.exists(path): continue

        df = pd.read_csv(path)
        y_true = (df['label'] == 'dga').astype(int)
        y_pred = df['pred']

        acc.append(accuracy_score(y_true, y_pred))
        pre.append(precision_score(y_true, y_pred, zero_division=0))
        rec.append(recall_score(y_true, y_pred, zero_division=0))
        f1.append(f1_score(y_true, y_pred, zero_division=0))

        fpr_val, tpr_val = fpr_tpr(y_true, y_pred)
        fpr.append(fpr_val)
        tpr.append(tpr_val)
        if 'query_time' in df.columns: qt.append(df['query_time'].mean())

    if acc:
        print(f'{family.split(".")[0]:15}: '
              f'acc:{np.mean(acc):.2f}±{np.std(acc):.3f} '
              f'f1:{np.mean(f1):.2f}±{np.std(f1):.3f} '
              f'pre:{np.mean(pre):.2f}±{np.std(pre):.3f} '
              f'rec:{np.mean(rec):.2f}±{np.std(rec):.3f} '
              f'FPR:{np.mean(fpr):.2f}±{np.std(fpr):.3f} '
              f'TPR:{np.mean(tpr):.2f}±{np.std(tpr):.3f} '
              f'QT:{np.mean(qt):.5f}±{np.std(qt):.5f}')

        all_acc.append(np.mean(acc))
        all_acc_std.append(np.std(acc))
        all_pre.append(np.mean(pre))
        all_pre_std.append(np.std(pre))
        all_rec.append(np.mean(rec))
        all_rec_std.append(np.std(rec))
        all_f1.append(np.mean(f1))
        all_f1_std.append(np.std(f1))
        all_fpr.append(np.mean(fpr))
        all_fpr_std.append(np.std(fpr))
        all_tpr.append(np.mean(tpr))
        all_tpr_std.append(np.std(tpr))
        all_qt.append(np.mean(qt))
        all_qts.append(np.std(qt))
        total_unknowns_global += total_unknowns

print("\n### ✅ Métricas recolectadas correctamente ###")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
bazarbackdoor  : acc:0.81±0.039 f1:0.77±0.052 pre:0.92±0.045 rec:0.67±0.065 FPR:0.06±0.034 TPR:0.67±0.065 QT:0.00627±0.00081
bazarbackdoor_v2: acc:0.71±0.031 f1:0.62±0.053 pre:0.89±0.054 rec:0.48±0.060 FPR:0.06±0.034 TPR:0.48±0.060 QT:0.00616±0.00088
bazarbackdoor_v3: acc:0.48±0.017 f1:0.03±0.027 pre:0.19±0.180 rec:0.02±0.015 FPR:0.06±0.034 TPR:0.02±0.015 QT:0.00588±0.00074
bigviktor      : acc:0.52±0.024 f1:0.16±0.051 pre:0.63±0.156 rec:0.09±0.032 FPR:0.06±0.034 TPR:0.09±0.032 QT:0.00595±0.00071
bumblebee      : acc:0.88±0.033 f1:0.87±0.036 pre:0.93±0.037 rec:0.82±0.047 FPR:0.06±0.034 TPR:0.82±0.047 QT:0.00600±0.00072
ccleaner       : acc:0.89±0.035 f1:0.88±0.040 pre:0.94±0.036 rec:0.83±0.059 FPR:0.06±0.034 TPR:0.83±0.059 QT:0.00639±0.00077
dmsniff        : acc:0.90±0.093 f1:0.89±0.136 pre:0.94±0.035 rec:0.86±0.186 FPR:0.06±0.034 TPR:0.86±0.186 QT:0.00622±0.

In [39]:
import pandas as pd
import numpy as np

# Crear el DataFrame usando todas las listas de medias y desviaciones
metrics_dict = {
    'Family': [f.split('.')[0] for f in families],
    'Accuracy': all_acc,
    'Acc_std': all_acc_std,
    'Precision': all_pre,
    'Pre_std': all_pre_std,
    'Recall': all_rec,
    'Rec_std': all_rec_std,
    'F1-Score': all_f1,
    'F1_std': all_f1_std,
    'FPR': all_fpr,
    'FPR_std': all_fpr_std,
    'TPR': all_tpr,
    'TPR_std': all_tpr_std,
    'Query_Time_Mean': all_qt,
    'Query_Time_Std': all_qts
}

df_metrics = pd.DataFrame(metrics_dict)

# Calcular fila de promedios globales
global_means = df_metrics.mean(numeric_only=True).to_dict()
global_means['Family'] = 'GLOBAL_MEAN'

df_metrics = pd.concat([df_metrics, pd.DataFrame([global_means])], ignore_index=True)

output_path = f'/content/drive/My Drive/results/metricas_globales_final_{name_model}.csv'
df_metrics.to_csv(output_path, index=False)

print(f"✅ CSV final con todas las desviaciones guardado en: {output_path}")
display(df_metrics)

✅ CSV final con todas las desviaciones guardado en: /content/drive/My Drive/results/metricas_globales_final_bilbo.csv


,Family,Accuracy,Acc_std,Precision,Pre_std,Recall,Rec_std,F1-Score,F1_std,FPR,FPR_std,TPR,TPR_std,Query_Time_Mean,Query_Time_Std
0,bazarbackdoor,0.806000,0.039294,0.921140,0.045265,0.670000,0.064653,0.774073,0.051724,0.058,0.03439,0.670000,0.064653,0.006273,0.000806
1,bazarbackdoor_v2,0.708667,0.031170,0.894817,0.053907,0.475333,0.060373,0.617996,0.053209,0.058,0.03439,0.475333,0.060373,0.006156,0.000884
2,bazarbackdoor_v3,0.478667,0.017461,0.188704,0.179680,0.015333,0.015217,0.028051,0.027446,0.058,0.03439,0.015333,0.015217,0.005884,0.000739
3,bigviktor,0.517333,0.023655,0.629761,0.156247,0.092667,0.031616,0.159833,0.050978,0.058,0.03439,0.092667,0.031616,0.005952,0.000708
4,bumblebee,0.881000,0.032899,0.934611,0.036564,0.820000,0.047046,0.872875,0.036343,0.058,0.03439,0.820000,0.047046,0.005996,0.000721
5,ccleaner,0.885667,0.035373,0.935593,0.036148,0.829333,0.058818,0.878042,0.039582,0.058,0.03439,0.829333,0.058818,0.006388,0.000770
6,dmsniff,0.902667,0.093093,0.935633,0.034826,0.863333,0.185766,0.885668,0.136175,0.058,0.03439,0.863333,0.185766,0.006221,0.000786
7,enviserv,0.898333,0.029562,0.937412,0.035342,0.854667,0.044101,0.893403,0.031653,0.058,0.03439,0.854667,0.044101,0.006162,0.001106
8,fosniw,0.471000,0.017195,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.058,0.03439,0.000000,0.000000,0.006103,0.000874
9,goz,0.971000,0.017195,0.946150,0.029872,1.000000,0.000000,0.972084,0.015992,0.058,0.03439,1.000000,0.000000,0.006206,0.000990
